# CSC 4792 Mini Project — Chama Town Council Dataset (Project Team #29)

This notebook documents the full pipeline used to build the dataset for
**Chama Town Council**, from web scraping and PDF extraction through to
privacy-safe data cleaning and codebook generation.

**Team:** Project Team #29
**Council:** Chama Town Council, Eastern Province, Zambia
**Council website:** https://www.chamacouncil.gov.zm

## Pipeline overview

1. **Web scraping** — CDF-related news posts and key pages (CDF Tracker,
   Constituency pages, District profile) from the council website.
2. **PDF extraction** — council budgets (2023, 2024), the District
   Integrated Development Plan (IDP), and CDF community project documents.
3. **Data cleaning** — turning the raw scraped/extracted material into
   structured, analysis-ready CSVs — with a deliberate step to exclude
   any personally identifiable data (see the Privacy section below).
4. **Codebook generation** — documenting every column in the final dataset.

Each stage below calls the actual script used to produce that stage's
output, so this notebook is a faithful record of how the dataset was
built, not just a description of it.


In [1]:
import sys
from pathlib import Path

# Make the scripts/ folder importable from this notebook
sys.path.append(str(Path("..") / "scripts"))

import pandas as pd


## 1. Web scraping

`scripts/web_scraper.py` politely scrapes the Chama Town Council website
for CDF-related content.

**Methodology notes:**
- The site has **no `robots.txt` file** (confirmed via a direct request,
  which returned a 404). This means there are no published crawling
  rules to follow — we still scrape responsibly by identifying ourselves
  with a descriptive `User-Agent`, adding a delay between requests, and
  targeting only relevant pages.
- The site's SSL certificate fails validation ("not within its validity
  period") even from a machine with a correct system clock — this
  appears to be a misconfiguration on the council's own server, common
  among smaller local government sites in Zambia. Certificate
  verification is deliberately disabled *only* for this known,
  specific government domain, and that decision is documented in the
  script itself.
- The site has **no sitemap**, and its pagination (`?paged=N`, `/page/N/`)
  does not surface older news posts beyond the first page. Post
  discovery therefore combines a handful of manually identified pages
  with whatever the homepage's first page of links reveals.

Running this cell re-scrapes the site live. It requires an internet
connection and may take a few minutes due to the polite per-request delay.


In [2]:
from web_scraper import main as run_web_scraper

run_web_scraper()


Step 1: Trying to discover post URLs via sitemap...
  [!] Failed to fetch https://www.chamacouncil.gov.zm/sitemap.xml: 404 Client Error: Not Found for url: https://www.chamacouncil.gov.zm/sitemap.xml
  [!] Failed to fetch https://www.chamacouncil.gov.zm/sitemap_index.xml: 404 Client Error: Not Found for url: https://www.chamacouncil.gov.zm/sitemap_index.xml
  [!] Failed to fetch https://www.chamacouncil.gov.zm/wp-sitemap.xml: 404 Client Error: Not Found for url: https://www.chamacouncil.gov.zm/wp-sitemap.xml
Step 2: Discovering post URLs via homepage pagination...
  [+] Page 1: 4 post links found, 4 new
  [+] Page 2: 4 post links found, 0 new
  [+] No new posts on page 2 — stopping pagination
Step 3: Combining all discovered + manually identified pages...
  [+] Total unique URLs to scrape: 12
Step 4: Fetching and parsing each page...
  (1/12) https://www.chamacouncil.gov.zm/?p=2029
  (2/12) https://www.chamacouncil.gov.zm/?p=2364
  (3/12) https://www.chamacouncil.gov.zm/?p=2438
  (4/12

## 2. PDF extraction

`scripts/pdf_extractor.py` downloads and extracts tabular/text data from
the council's PDF documents: the 2023 and 2024 Output-Based Budgets, the
District Integrated Development Plan (IDP), and several CDF community
project / empowerment loan / empowerment grant documents.

**Methodology notes:**
- Table extraction is attempted first (via `pdfplumber`'s table
  detection); pages with no detectable table structure fall back to
  plain text extraction, so no page's content is silently dropped.
- **Five of the CDF empowerment loan/grant PDFs for 2025 returned zero
  extractable content**, even with the text fallback. Inspecting them
  shows every page reads only "CamScanner" — these are scanned images
  with no underlying text layer at all, not PDFs with a missing table
  structure. Extracting their content would require OCR, which was out
  of scope given the project timeline; this is a genuine, acknowledged
  gap in the dataset rather than an extraction failure.
- The council server is slow and occasionally times out on larger
  files; downloads are streamed with a generous timeout and automatic
  retries to handle this reliably.


In [3]:
from pdf_extractor import main as run_pdf_extractor

run_pdf_extractor()


Step 1: Downloading 12 PDFs...
  [=] Already downloaded: Chama-Town-Council-2024-Output-Based-Budget.pdf
  [=] Already downloaded: Chama-Town-Council-2023-Budget.pdf
  [=] Already downloaded: CHAMA-DISTRICT-INTEGRATED-DEVELOPMENT-PLAN-28_02_2025.pdf
  [=] Already downloaded: 2025-Approved-Community-Projects-Chama-South-Constituency.pdf
  [=] Already downloaded: 2025-Community-Projects-Proposed-Chama-South-Constituency.pdf
  [=] Already downloaded: 2025-Approved-Loans-Chama-South.pdf
  [=] Already downloaded: 2025-Approved-Grants-Chama-South.pdf
  [=] Already downloaded: Chama-South-Community-Projects.pdf
  [=] Already downloaded: Chama-South-Constituency-Empowerment-Loans-2024.pdf
  [=] Already downloaded: Chama-South-Constituency-Empowerment-Grant-2024.pdf
  [=] Already downloaded: 2025-Approved-Loans-Chama-North.pdf
  [=] Already downloaded: 2025-Approved-Grants-Chama-North.pdf
Step 2: Extracting tables from each PDF...
  Processing Chama-Town-Council-2024-Output-Based-Budget.pdf...


## 3. Data cleaning

`scripts/data_cleaning.py` turns the raw scraped/extracted material into
structured, analysis-ready CSVs.

### Privacy: personally identifiable data was deliberately excluded

Several source pages and PDFs mix **project-level data** (safe to
publish — project names, sectors, wards, funding amounts) with
**person-level data**: CDF Empowerment Grant/Loan recipient names, and —
most importantly — **the names of individual school children** receiving
Secondary Boarding Bursaries and Skills Development Bursaries, alongside
their ward, sex, and grade.

Even though the council itself published this on a public web page, a
dataset that lists identifiable children by name has no place in a
published, downloadable Kaggle dataset. This connects directly to
material covered earlier in this course on the Zambian Data Protection
Act (2021).

The cleaning script enforces this by **only ever reading text that sits
between a "Community Projects" marker and whatever section comes next**
in the source text — it never reads into the Empowerment Grants,
Empowerment Loans, or Bursaries sections at all, on any page, regardless
of which page it came from. This is a structural safeguard, not a
post-hoc filter: the personal-data sections are never parsed in the
first place.

### Other cleaning notes

- The CDF Tracker / Constituency page text has no consistent separator
  between a project's name and its description, so these are kept
  combined in one field (`project_name_and_description`) rather than
  split inaccurately.
- Because the source text also has no clean marker between one
  project's ward/location and the next project's number, a small number
  of rows have a `ward_location_raw` value that includes a fragment of
  the next project's leading text. This is a known, minor limitation of
  parsing free-form government web text with regular expressions.
- IDP tables (population, revenue forecast) required filtering out
  repeated header rows and, in one case, a leftover historical (2019-
  2024) table that happened to sit on the same PDF page as the 2025-
  2030 forecast table we were extracting.


In [4]:
from data_cleaning import main as run_data_cleaning

run_data_cleaning()

Cleaning: CDF Tracker + Constituency pages, Community Projects only (privacy-safe)
  [+] Saved 92 rows -> data\processed\db-unza26-csc4792-cdf_community_projects.csv
Cleaning: Chama South Community Projects (PDF)
  [+] Saved 16 rows -> data\processed\db-unza26-csc4792-chama_south_community_projects.csv
Cleaning: IDP population table
  [+] Saved 27 rows -> data\processed\db-unza26-csc4792-idp_ward_population.csv
Cleaning: IDP revenue forecast table
  [+] Saved 50 rows -> data\processed\db-unza26-csc4792-idp_revenue_forecast.csv


## 4. Codebook generation

`scripts/build_codebook.py` documents every column across the four
processed dataset files: its meaning, data type, and an example value.


In [5]:
from build_codebook import build_codebook

build_codebook()


Building codebook
  [+] Saved 22 rows -> data\processed\db-unza26-csc4792-codebook.csv


## 5. Preview of the final processed dataset

A quick look at each processed file to confirm the pipeline produced
clean, structured output.


In [6]:
processed_dir = Path("..") / "data" / "processed"

cdf_projects = pd.read_csv(processed_dir / "db-unza26-csc4792-cdf_community_projects.csv", sep="|")
cdf_projects.head(10)


,source_url,year,project_no,project_name_and_description,sector,type,ward_location_raw
0,https://www.chamacouncil.gov.zm/?page_id=1127,2022,1.0,PROCUREMENT OF SCHOOL DESKS FOR CHAMA DAY AND ...,EDUCATION,procurement,Kampemba and Kaozi CHAMA DAY SEC AND KASANGANI
1,https://www.chamacouncil.gov.zm/?page_id=1127,2022,2.0,MATRESSES (4INCHES) FOR HEALTH POSTS procureme...,HEALTH,procurement,ALL WARDS ALL 23 HEALTH POST
2,https://www.chamacouncil.gov.zm/?page_id=1127,2022,3.0,CLASS ROOM BLOCK AT CHIKONTHA PRIMARY Construc...,EDUCATION,Construction,Chisunga Chikontha
3,https://www.chamacouncil.gov.zm/?page_id=1127,2022,4.0,BANANA BOAT FOR KAKOMA TO MULILO CROSSING POIN...,TRANSPORATION,procurement,Chisunga KAKOMA
4,https://www.chamacouncil.gov.zm/?page_id=1127,2022,5.0,WILA TO KAKOMA PRIMARY SCHOOL CROSSING POINTS ...,INFRASTRUCTURE,Construction,Chisunga KAKOMA-WILA 6 DRILLING OF 8 BOREHOLES...
5,https://www.chamacouncil.gov.zm/?page_id=1127,2022,8.0,HAND PUMPS-LOT 3,WATER AND SANITATION,DRILLING,Kamphemba MAKENI B 7 DRILLING OF 8 BOREHOLES D...
6,https://www.chamacouncil.gov.zm/?page_id=1127,2022,8.0,HAND PUMPS-LOT 2,WATER AND SANITATION,DRILLING,"Kamphemba, Luangwa MAKENI B AND KASEMPWE 8 DRI..."
7,https://www.chamacouncil.gov.zm/?page_id=1127,2022,6.0,HAND PUMPS -LOT 4,WATER AND SANITATION,DRILLING,kaozi ZONE 1 9 DRILLING OF
8,https://www.chamacouncil.gov.zm/?page_id=1127,2022,8.0,BOREHOLES Drilling and Installation of 8 hand ...,WATER AND SANITATION,DRILLING,"Mphalausenga, Luangwa, Kalinkhu GOMBE, KOMBAZI..."
9,https://www.chamacouncil.gov.zm/?page_id=1127,2022,10.0,CLASS ROOM BLOCK AT GOMBE PRIMARY CONSTRUCTION...,EDUCATION,CONSTRUCTION,KALINKHU Gombe


In [7]:
south_projects = pd.read_csv(processed_dir / "db-unza26-csc4792-chama_south_community_projects.csv", sep="|")
south_projects


,project_no,description,ward,amount_kwacha,sector,year,constituency
0,Project 1,FUEL AND MAINTENANCE OF EARTH MOVING MACHINES ...,All Wards,"3,500,000.00",ROADS,2024,south
1,Project 2,JOINT CONTRIBUTION HIRING WITH CHAMA NORTH CDF...,All Wards,"250,000.00",ROADS,2024,south
2,Project 3,JOINT PROCUREMENT OF EARTH MOVING MACHINES (WA...,All Wards,"5,000,000.00",ROADS,2024,south
3,Project 4,AMBULANCE FOR HEALTH DEPARTMENT,All Wards,"2,700,000.00",HEALTH,2024,south
4,Project 5,MOTORBIKES FOR THE THREE CHIEF RETAINERS,3 CHIEFDOMS,"75,000.00",TRADITION,2024,south
5,Project 6,CONSTRUCTION OF AN ABLUTION BLOCK FOR THE GIRL...,CHILUBALUBA,"600,000.00",EDUCATION,2024,south
6,Project 7,CONSTRUCTION OF A HEALTH POST AT MUTEMBWA HEAL...,LUNZI,"1,798,973.80",HEALTH,2024,south
7,Project 8,CONSTRUCTION OF ONE STAFF HOUSE AT ZOOLE HEALT...,LUMEZI,"650,000.00",HEALTH,2024,south
8,Project 9,CONSTRUCTION A STAFF HOUSE AT CHANGOZI PRIMARY...,CHILUBALUBA,"649,757.14",EDUCATION,2024,south
9,Project 10,PROCUREMENT OF 200 MATTRESSES FOR CHAMA SOUTH ...,VILIMUKULU & LUNZI,"317,620.00",EDUCATION,2024,south


In [11]:
ward_population = pd.read_csv(processed_dir / "db-unza26-csc4792-idp_ward_population.csv", sep="|")
ward_population


,area_name,total_both_sexes,total_male,total_female,rural_both_sexes,rural_male,rural_female,urban_both_sexes,urban_male,urban_female,area_km2,population_density
0,CHAMA DISTRICT,"140,784","68,978","71,806","127,861","62,770","65,091","12,923","6,208","6,715","17,472.8",8.10
1,Chama North,"83,781","41,063","42,718","70,858","34,855","36,003","12,923","6,208","6,715",10823.7,7.74
2,Chisunga,"4,686","2,334","2,352","4,686","2,334","2,352",-,-,-,1555.5,3.01
3,Kalinkhu,"7,280","3,517","3,763","7,280","3,517","3,763",-,-,-,406.2,17.92
4,Kamphemba,"17,392","8,416","8,976","4,469","2,208","2,261","12,923","6,208","6,715",72.1,241.22
5,Kaozi,"8,908","4,341","4,567","8,908","4,341","4,567",-,-,-,184.8,48.20
6,Luangwa,"3,947","1,987","1,960","3,947","1,987","1,960",-,-,-,1042.2,3.79
7,Manthepa,"2,033",983,"1,050","2,033",983,"1,050",-,-,-,151.4,13.43
8,Mazonde,"1,755",880,875,"1,755",880,875,-,-,-,1154.7,1.52
9,Mbazi,"5,932","2,948","2,984","5,932","2,948","2,984",-,-,-,175.3,33.84


In [9]:
revenue_forecast = pd.read_csv(processed_dir / "db-unza26-csc4792-idp_revenue_forecast.csv", sep="|")
revenue_forecast.head(10)


,revenue_item,year_2025,year_2026,year_2027,year_2028,year_2029,year_2030
0,Residential,25200,26964.00,28851.4800,30871.08360,33032.05945,35344.30361
1,Industrial,7650,8185.50,8758.4850,9371.57895,10027.58948,10729.52074
2,Subitem total,32850,35149.50,37609.9650,40242.66255,43059.64893,46073.82435
3,Personal levy,43995,47074.65,50369.8755,53895.76679,57668.47046,61705.26339
4,Subitem Total,43995,47074.65,50369.8755,53895.76679,57668.47046,61705.26339
5,Plan scrutiny fee,45250,48417.50,51806.7250,55433.19575,59313.51945,63465.46581
6,Rentals/lease of Council’s\nproperties,106000,113420.00,121359.4000,129854.55800,138944.37710,148670.48350
7,Non-Land Application form fees,35600,38092.00,40758.4400,43611.53080,46664.33796,49930.84161
8,Market fees,109500,117165.00,125366.5500,134142.20850,143532.16310,153579.41450
9,"Loading fees (buses, trucks,\ntrains, taxies e...",86720,92790.40,99285.7280,106235.72900,113672.23000,121629.28610


In [10]:
codebook = pd.read_csv(processed_dir / "db-unza26-csc4792-codebook.csv", sep="|")
codebook


,file,column,description,data_type,example_value
0,db-unza26-csc4792-cdf_community_projects.csv,source_url,URL of the Chama Town Council website page the...,text (URL),https://www.chamacouncil.gov.zm/?page_id=1127
1,db-unza26-csc4792-cdf_community_projects.csv,year,"The CDF funding year the project relates to, a...",integer (year),2023
2,db-unza26-csc4792-cdf_community_projects.csv,project_no,The project's listed number on the source page...,text,3
3,db-unza26-csc4792-cdf_community_projects.csv,project_name_and_description,"The project's name and description, combined i...",text,CLASS ROOM BLOCK AT CHIKONTHA PRIMARY Construc...
4,db-unza26-csc4792-cdf_community_projects.csv,sector,The project's sector classification as listed ...,categorical text,EDUCATION
5,db-unza26-csc4792-cdf_community_projects.csv,type,The type of activity carried out for the proje...,categorical text,Construction
6,db-unza26-csc4792-cdf_community_projects.csv,ward_location_raw,The ward and/or site location of the project. ...,text,Chisunga Chikontha
7,db-unza26-csc4792-chama_south_community_projec...,project_no,Project identifier as listed in the official C...,text,Project 4
8,db-unza26-csc4792-chama_south_community_projec...,description,Full description of the approved CDF project.,text,AMBULANCE FOR HEALTH DEPARTMENT
9,db-unza26-csc4792-chama_south_community_projec...,ward,"Ward, or wards, the project serves. Some proje...",text,LUNZI


## Summary of known limitations

For transparency, and per the Data in Brief methodology expectations,
the following limitations of this dataset are acknowledged:

1. **No robots.txt existed** on the source site to confirm formal
   crawling permissions; the site was scraped respectfully in its
   absence (identified User-Agent, request delays, targeted pages only).
2. **SSL certificate verification was disabled** for this one known
   government domain due to a certificate misconfiguration on the
   council's own server — a deliberate, documented judgement call.
3. **No sitemap exists**, and pagination did not surface the site's
   full historical news archive; the dataset includes the news posts
   that were discoverable via the homepage and known direct links, not
   necessarily every CDF-related post the council has ever published.
4. **Five 2025 CDF empowerment loan/grant PDFs were scanned images**
   with no extractable text (OCR was out of scope for this project) and
   are therefore not represented in the structured dataset.
5. **Personally identifiable information was deliberately excluded**
   even where the council published it publicly — specifically,
   individual bursary recipients (including school children) and
   individual empowerment loan/grant recipients. Only project-level and
   aggregate data is included.
6. **Some free-text fields could not be cleanly split** (project name
   vs. description; ward/location vs. the next project's lead-in text)
   because the source website text has no consistent delimiter marking
   these boundaries.
